# Two-Echelon EVRP

**What is two-echelon?** Goods leave a **central depot** (first echelon / long haul), are transferred at **satellite** hubs, then **last-mile** vehicles serve customers from those hubs (second echelon). Your benchmark encodes depot, satellites, customers, and chargers on the same road graph.

**What you configure**
- **`G`** — city road network.
- **Central depot** — `depot_lat`, `depot_lon`.
- **Satellites** — `satellite_locations`: list of `(lat, lon)` transfer points (snap to roads like the depot).
- **Customers** — generated (`num_customers`) **or** CSV (`customer_csv_path`, then `num_customers` is `None`).
- After customers exist, **`assign_customers_to_satellites`** links each customer to the nearest satellite (great-circle), then capacities are adjusted so the second echelon stays feasible as a benchmark.

**`generation_context`** carries state between pipeline steps until **`finalize`** produces **`instance`**.

In [ ]:
# Prefer: pip install -e . (repo root) in this kernel. Fallback if import fails:
import sys
from pathlib import Path
for _p in (Path.cwd() / "src", Path.cwd().parent / "src"):
    if _p.is_dir() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

# Use the project environment; from the repo root run: pip install -e .
import evrp_instance_generator_framework as evrp
from evrp_instance_generator_framework.export.instance_export import export_instance
# Folium helpers (import from submodule so notebooks work even if the top-level namespace is stale)
from evrp_instance_generator_framework.visualization import (
    map_benchmark_interactive,
    map_city_roads_interactive,
    map_city_roads_with_depot_interactive,
)
from evrp_instance_generator_framework.variants import two_echelon as te_variant  # satellites + 2E logic

print("Using package from:", evrp.__file__)
print("Interactive map helpers loaded:", map_city_roads_interactive.__name__)


In [ ]:
# Reliable map rendering in most notebook frontends:
%matplotlib inline

# Optional interactive mode (pan/zoom) if supported by your frontend:
# %pip install ipympl
# %matplotlib widget

## City and road network

Same as other notebooks: **`G`** is the OSM-based driving graph for routing and snapping.

In [ ]:
city = "Casablanca"
country = "Morocco"
G = evrp.prepare_city_road_network(city, country, flat_terrain=True)
map_city_roads_interactive(G)


## Central depot and satellites

- **`depot_lat`, `depot_lon`:** main warehouse (first-echelon origin).
- **`satellite_locations`:** each tuple is a **transfer hub** where vehicles reload or hand off to second-echelon routes. Use coordinates that make sense geographically (suburbs, logistics zones). They are snapped to legal road nodes internally.

The quick plot below shows only the central depot marker; satellites appear in the final map after generation.

In [ ]:
depot_lat = 33.573
depot_lon = -7.590
satellite_locations = [(33.595, -7.620), (33.552, -7.575), (33.542, -7.618)]
map_city_roads_with_depot_interactive(G, depot_lat, depot_lon)


## Customer count (if not using CSV)

Used only when `customer_csv_path` is `None`.

In [ ]:
num_customers = 16


## Charging stations count

Public charging facilities for EV routing/energy modelling—selected after customers (and satellite assignment) exist.

In [ ]:
num_stations = 5


## EV features

Vehicle physics shared with other variants (battery, mass, drag, …) for feasibility reports.

In [ ]:
ev_features = evrp.EVFeatures(battery_capacity_kwh=75.0, driver_behavior="passive")


## Optional customer CSV

Either synthetic customers or your own CSV (same schema as exports). CSV must align with graph **`G`** (`movement_node_id`, etc.).

In [ ]:
customer_csv_path = None  # example: "my_customers.csv"


## Run generation (two-echelon pipeline)

Order of operations in the code cell:

1. **`prepare_graph_and_depot`** — graph + snapped central depot + depot travel times.
2. **`setup_satellites`** — snap and create satellite records from `satellite_locations`.
3. **`generate_customers`** — synthetic list **or** CSV from config.
4. **`assign_customers_to_satellites`** — each customer → nearest satellite; capacities updated.
5. **`generate_stations`** — charging stations.
6. **`finalize`** — full **`instance`** with two-echelon metadata and feasibility.

`generation_context` is the working object passed through steps 1–5.

In [ ]:
if "customer_csv_path" not in globals():
    customer_csv_path = None
config = evrp.GenerationConfig(
    variant="two_echelon_evrp",
    city=city,
    country=country,
    depot_lat=depot_lat,
    depot_lon=depot_lon,
    satellite_locations=tuple(satellite_locations),
    seed=42,
    num_customers=num_customers if customer_csv_path is None else None,
    customer_csv_path=customer_csv_path,
    num_stations=num_stations,
    customer_pattern="rc",
    time_window_tightness="medium",
)
generation_context = te_variant.prepare_graph_and_depot(config, ev_features, movement_graph=G)
generation_context = te_variant.setup_satellites(generation_context)
generation_context = te_variant.generate_customers(generation_context)
generation_context = te_variant.assign_customers_to_satellites(generation_context)  # nearest hub per customer
generation_context = te_variant.generate_stations(generation_context)
instance = te_variant.finalize(generation_context, compute_matrices=False, run_energy_feasibility=True)
map_benchmark_interactive(instance)


When using CSV, columns must match (names and types):  
`id`, `lat`, `lon`, `movement_node_id`, `snap_distance_m`, `demand`, `service_time_s`, `parking_time_s`, `time_open_s`, `time_close_s`.

Tip: export a generated instance once and reuse its customer table as a template.

## Export

Saves the benchmark (JSON below) so you can load it elsewhere or attach it to papers/experiments.

In [ ]:
out_dir = export_instance(instance, output_dir="benchmark_export_two_echelon", fmt="json")
print("Exported to:", out_dir)
